In [1]:
# Calculating country level population data

In [2]:
import os
import xarray as xr
import numpy as np

In [3]:
# === Path config ===
POP_DIR = "/glade/work/awells/air_quality/SSP_pop/SSP2/"
MASKS_DIR = "/glade/work/awells/air_quality/BMR/masks/country/"

In [4]:
# Use country masks to create a BMR mask for each country
mask_file = "GBD_Country_Masks_0.10.nc"
mask_path = os.path.join(MASKS_DIR, mask_file)
masks = xr.open_dataarray(mask_path)

pop_file = "ssp2_coarse_grid_annual_2000-2100.nc"
pop_path = os.path.join(POP_DIR, pop_file)
pop = xr.open_dataarray(pop_path)

# Adjust indices to match (with small tolerance)
# e.g., max 1e-7 km distance
pop = pop.reindex_like(masks, method="nearest", tolerance=1e-9)

In [6]:
# Loop over countries, sum the population for each country and apply to list
population_by_country = []
for i in range(len(masks.country)):
    print(i)
    mask = masks.isel(country=i)
    country = masks.isel(country=i)["country"]
    pop_country = (xr.where(
        mask == 1,
        pop,
        np.nan)).sum(dim=("lat", "lon"))
    population_by_country.append(pop_country)

pop_array = xr.concat(population_by_country, "country")

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203


In [12]:
# Save country level population
description = ("Country level population sum for years 2000-2100 "
               "- scripts by A.F. Wells (2025)")

pop_array.attrs["description"] = description

out_file = "ssp2_country_level_2000-2100.nc"
out_path = os.path.join(POP_DIR, out_file)
pop_array.to_netcdf(out_path)